In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import numpy as np
import scanpy as sc
from tqdm.auto import tqdm
from descope.tokenizer import tokenize_adata_to_hf_dataset_for_rna

In [ ]:
def generate_condition_col_for_tahoe100m(adata_path: str) -> sc.AnnData:
    def _preprocess(x: str) -> str:
        import ast
        x: tuple = ast.literal_eval(x)[0]
        drug = x[0].strip()
        dose = float(x[1])
        norm_dose = float(dose / 5.0)  # max_dose in tahoe-100m: 5μM
        return f"{drug}_{norm_dose}"
    
    adata = sc.read_h5ad(adata_path)
    adata.obs["drug"] = adata.obs["drug"].str.strip()
    adata = adata[~adata.obs["drug"].isin(['Sacubitril/Valsartan', 'Verteporfin'])].copy()
    adata.obs["condition"] = adata.obs["drugname_drugconc"].apply(_preprocess)
    return adata


def split_anndata_into_n_chunks(adata: sc.AnnData, n_chunks: int = 10) -> list[sc.AnnData]:
    indices = np.array_split(np.arange(adata.n_obs), n_chunks)
    return [adata[idx] for idx in indices]

In [ ]:
data_dir = "/fse/home/wupengpeng/perturbation_datasets/origin_datasets/Tahoe_100M"
h5ads = [os.path.join(data_dir, h5ad) for h5ad in os.listdir(data_dir)]
h5ads

In [ ]:
for h5ad in tqdm(h5ads, desc="tokenize all"):
    plate = h5ad.split("/")[-1].split("_")[0]

    # raw counts
    adata = generate_condition_col_for_tahoe100m(h5ad)
    
    # split into chunks
    # If memory is insufficient to convert adata_chunk.X to a dense matrix, increase n_chunks
    for i, adata_chunk in enumerate(tqdm(split_anndata_into_n_chunks(adata, n_chunks=20), desc="tokenize chunks")):
        tokenize_adata_to_hf_dataset_for_rna(
            adata=adata_chunk,
            cell_line_col="cell_line",
            target_sum=1e4,
            pert_col="condition",
            ctrl_name="DMSO_TF_0.0",
            skip_raw_counts_check=True,
            save_dir=f"/fse/home/wupengpeng/perturbation_datasets/origin_datasets/Tahoe_100M_Tokenized/{plate}_chunk{i}"
        )